# Raw EDA และ pipeline preview: dim_customer

ลำดับ: EDA raw source → Landing star schema (ค่าต้นทางและ dim IDs) → Staging monadsquishy → Integration พร้อมใช้ → Metabase

Notebook นี้ใช้ transformation เดียวกับ flow แต่ไม่เขียนหรือ publish ข้อมูล และไม่ต้องมี snapshot จาก pipeline ก่อน

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "eda.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils" / "eda.py").is_file():
    raise RuntimeError("เปิด Jupyter จากโฟลเดอร์โปรเจกต์หรือ notebook")
sys.path.insert(0, str(PROJECT_ROOT))
from utils.eda import load_raw_inputs, landing_preview, staging_preview, integration_preview
from utils.schema import MODEL_KEYS


## 1. EDA จาก raw source ก่อนแปลงข้อมูล

ระบุไฟล์ SQLite ที่มีอยู่ใน RAW_DB_PATHS หรือใช้ฐานข้อมูลตัวอย่างสาธารณะเมื่อเว้นว่าง การอ่านไม่เปลี่ยนค่าหรือชื่อคอลัมน์ต้นทาง

In [ ]:
TABLE = "dim_customer"
SOURCE_SYSTEMS = ("chinook", "northwind")
RAW_DB_PATHS = {}  # ตัวอย่าง: {"chinook": "/path/to/chinook.sqlite", "northwind": "/path/to/northwind.db"}
raw_inputs = load_raw_inputs(TABLE, db_paths=RAW_DB_PATHS, sources=SOURCE_SYSTEMS)
for name, raw in raw_inputs.items():
    print({"source_table": name, "rows": len(raw), "columns": len(raw.columns)})
    display(raw.head(5))
    profile = pd.DataFrame({"dtype": raw.dtypes.astype(str), "null_count": raw.isna().sum(),
                            "distinct_count": raw.nunique(dropna=True)})
    display(profile)
    display(raw.describe(include="all"))


## 2. Landing: Kimball star schema ด้วย raw values

เปลี่ยนโครงสร้างและสร้าง technical keys ได้ แต่ยังไม่ normalize ข้อความ ตรวจเบอร์โทร หรือ parse วันที่ การ join ต้องคงจำนวนแถว fact; date_id และ dim IDs ถูกเชื่อมจาก dimensions

Calendar attributes จะถูกเติมใน Staging หลังตรวจวันที่แล้ว

In [ ]:
landing_models = landing_preview(TABLE, raw_inputs)
landing = {source: bundle[TABLE] for source, bundle in landing_models.items()}
for source, frame in landing.items():
    print({"zone": "landing", "source": source, "rows": len(frame)})
    display(frame.head(10))
    key = MODEL_KEYS[TABLE]
    print({"primary_key": key, "null_keys": int(frame[key].isna().sum()),
           "duplicate_keys": int(frame[key].dropna().duplicated().sum())})
    if TABLE == "fact_order":
        display(frame[["order_line_id", "customer_id", "employee_id", "product_id", "date_id",
                       "customer_location_id", "employee_location_id"]].head(10))


## 3. Staging: monadsquishy

ใช้ rules เดียวกับ production flow แก้รูปแบบและตรวจคุณภาพตรงนี้ ผล preview ยังเก็บแถวที่ผิดไว้ให้ตรวจสอบ; production quality gate จะไม่ publish แถว fact ที่ผิด

In [ ]:
stage_result = staging_preview(TABLE, landing_models)
staging = stage_result["staging"]
metadata = stage_result["metadata"]
for source, frame in staging.items():
    print({"zone": "staging", "source": source, "rows": len(frame)})
    display(frame.head(10))
display(metadata)
if not stage_result["relationships"].empty:
    display(stage_result["relationships"])
if not metadata.empty:
    display(metadata.loc[metadata["status"].eq("failed")])


## 4. Integration พร้อมใช้ และ Metabase

รวมทั้งสอง source เป็นตารางเดียวหลัง Staging ใช้ dim IDs join กับ fact ได้เลย Production จะตรวจ PK/FK และ quality gate ก่อน publish และส่ง snapshot ที่สำเร็จเข้า PostgreSQL สำหรับ Metabase

In [ ]:
integration = integration_preview(TABLE, staging)
print({"zone": "integration", "table": TABLE, "rows": len(integration)})
display(integration.head(10))
display(integration.describe(include="all"))
if TABLE == "fact_order":
    display(integration.groupby("source_system", dropna=False).agg(
        order_line_count=("order_line_id", "count"), total_quantity=("quantity", "sum"), total_amount=("amount", "sum")))
    display(integration["quality_status"].value_counts(dropna=False).to_frame("row_count"))


## ข้อสรุปก่อนนำ rules ไปใช้ใน flow

- Grain และ primary key ของตาราง:
- raw columns ที่มีปัญหา:
- Rules ที่ต้องปรับใน Staging:
- ความสัมพันธ์ dim/fact ที่ตรวจแล้ว:
- Metrics สำหรับ Metabase:

เมื่อ EDA และ review rules เสร็จ ให้รัน `python run_pipeline.py run` จากโฟลเดอร์โปรเจกต์ การ EDA เป็นขั้นตอนพัฒนา ไม่ถูกบังคับให้รัน notebook ใหม่ทุกครั้งใน production

ล้าง outputs ที่มีข้อมูลส่วนบุคคลหรือ secrets ก่อน commit